In [1]:

import os
from collections import defaultdict

import openslide
from openslide import OpenSlideError, OpenSlideUnsupportedFormatError


In [2]:

# Root directory containing .svs files (recursively scanned)
SVS_DIR = "/home/rapids/notebooks/slima/TGCA LUAD LUSC/TCGA LUAD LUSC"



In [3]:

def check_single_slide(slide_path):
    """
    Try to open a single .svs slide with OpenSlide.

    Returns:
        ("ok", None) if opened successfully
        ("error", (error_type, error_msg)) if an exception occurs
    """
    slide_path_str = str(slide_path)
    print(f"\n=== Processing slide: {slide_path_str} ===")

    try:
        slide = openslide.OpenSlide(slide_path_str)
        # If we got here, it opened fine
        print("[OK] Opened successfully")
        slide.close()
        return "ok", None

    except OpenSlideUnsupportedFormatError as e:
        print(f"[SKIP] Unsupported or missing WSI: {slide_path_str} ({e})")
        return "error", ("OpenSlideUnsupportedFormatError", str(e))

    except OpenSlideError as e:
        print(f"[SKIP] OpenSlideError on {slide_path_str}: {e}")
        return "error", ("OpenSlideError", str(e))

    except Exception as e:
        # Catch any unexpected errors
        print(f"[SKIP] Unexpected error opening {slide_path_str}: {e}")
        return "error", (type(e).__name__, str(e))


def scan_svs_directory(root_dir):
    """
    Recursively scan `root_dir` for .svs files, try to open each with OpenSlide,
    and collect statistics on successes and errors.
    """
    total_files = 0
    ok_files = 0
    error_files = 0

    # error_counts: map error_type -> count
    error_counts = defaultdict(int)

    # error_examples: map error_type -> list of (filepath, msg) (limited)
    error_examples = defaultdict(list)
    MAX_EXAMPLES_PER_ERROR = 5

    print(f"Scanning directory (recursively) for .svs files:\n  {root_dir}\n")

    for dirpath, dirnames, filenames in os.walk(root_dir):
        for fname in filenames:
            if not fname.lower().endswith(".svs"):
                continue

            total_files += 1
            fpath = os.path.join(dirpath, fname)

            status, err_info = check_single_slide(fpath)

            if status == "ok":
                ok_files += 1
            else:
                error_files += 1
                err_type, err_msg = err_info
                error_counts[err_type] += 1

                # Store a few example messages/files per error type
                if len(error_examples[err_type]) < MAX_EXAMPLES_PER_ERROR:
                    error_examples[err_type].append((fpath, err_msg))

    # Summary
    print("\n" + "=" * 70)
    print("SUMMARY OF .SVS VALIDATION")
    print("=" * 70)
    print(f"Root directory: {root_dir}")
    print(f"Total .svs files found: {total_files}")
    print(f"Successfully opened (good): {ok_files}")
    print(f"Failed to open (with errors): {error_files}")

    if error_files > 0:
        print("\nError breakdown by exception type:")
        for err_type, count in sorted(error_counts.items(), key=lambda x: x[0]):
            print(f"  - {err_type}: {count} file(s)")

        print("\nSample files and messages per error type (up to 5 each):")
        for err_type, examples in error_examples.items():
            print(f"\n>>> {err_type} (showing {len(examples)} example(s))")
            for fpath, msg in examples:
                print(f"    File: {fpath}")
                print(f"    Msg : {msg}")
    else:
        print("\nNo errors detected: all .svs files opened successfully.")

    print("\nDone.")



In [4]:



if __name__ == "__main__":
    scan_svs_directory(SVS_DIR)


Scanning directory (recursively) for .svs files:
  /home/rapids/notebooks/slima/TGCA LUAD LUSC/TCGA LUAD LUSC


=== Processing slide: /home/rapids/notebooks/slima/TGCA LUAD LUSC/TCGA LUAD LUSC/97dc4a13-09c6-4093-9b3d-becbbdd8944b/TCGA-90-A59Q-01Z-00-DX1.9F4ABA20-E9F7-4524-874E-E8C42D84AFFE.svs ===
[OK] Opened successfully

=== Processing slide: /home/rapids/notebooks/slima/TGCA LUAD LUSC/TCGA LUAD LUSC/86988f31-6d83-454b-badd-89d55db38fe5/TCGA-55-7728-01Z-00-DX1.1d47a5fe-cab5-4d5a-a62b-16f345334d25.svs ===
[OK] Opened successfully

=== Processing slide: /home/rapids/notebooks/slima/TGCA LUAD LUSC/TCGA LUAD LUSC/640fbebc-6d08-4d4a-bbc8-5d6d73626883/TCGA-49-4512-01Z-00-DX6.985c9088-ff83-4f13-8a95-cb55aa48682b.svs ===
[OK] Opened successfully

=== Processing slide: /home/rapids/notebooks/slima/TGCA LUAD LUSC/TCGA LUAD LUSC/8c4f94a6-f895-4e39-849a-87ba71a43549/TCGA-18-4086-01Z-00-DX1.1D06B771-F978-4763-8D4E-7B540E02C55A.svs ===
[OK] Opened successfully

=== Processing slide: /home/rapids/n